# Driver Safety AI - Training trên Google Colab\n
\n
Notebook này hướng dẫn train mô hình phát hiện hành vi lái xe nguy hiểm trên Google Colab với GPU miễn phí.

## 1. Setup môi trường và cài đặt thư viện

In [ ]:
# Kiểm tra GPU
!nvidia-smi
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.test.is_gpu_available()}")

In [11]:
# Cài đặt thư viện cần thiết
# !pip install -q tensorflow opencv-python pillow matplotlib pandas scikit-learn albumentations
!pip install -q 'tensorflow-cpu==2.15.0' 'protobuf>=4.25.3,<5' 'numpy<2.0'

ERROR: Could not find a version that satisfies the requirement tensorflow-cpu==2.15.0 (from versions: none)
ERROR: No matching distribution found for tensorflow-cpu==2.15.0


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import albumentations as A
from google.colab import files, drive
import zipfile
import shutil

## 2. Tải và chuẩn bị Dataset

In [ ]:
# Mount Google Drive để lưu trữ dữ liệu
drive.mount('/content/drive')

# Tạo thư mục làm việc
WORK_DIR = '/content/driver_safety'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
# Tải dataset mẫu (hoặc upload dataset của bạn)
# Option 1: Upload từ máy local
# uploaded = files.upload()

# Option 2: Tải dataset công khai
!wget -q https://example.com/driver_behavior_dataset.zip -O dataset.zip
!unzip -q dataset.zip

# Option 3: Sử dụng dataset từ Drive
# !cp /content/drive/MyDrive/datasets/driver_dataset.zip .
# !unzip -q driver_dataset.zip

In [ ]:
# Định nghĩa các class hành vi
CLASSES = [
    'normal',           # Lái xe bình thường
    'drowsy',          # Buồn ngủ
    'yawning',         # Ngáp
    'phone_call',      # Gọi điện thoại
    'texting',         # Nhắn tin
    'distracted',      # Mất tập trung
    'drinking',        # Uống nước
    'reaching_behind', # Với tay ra sau
    'no_seatbelt'      # Không thắt dây an toàn
]

NUM_CLASSES = len(CLASSES)
IMG_SIZE = (224, 224)  # Kích thước ảnh cho MobileNetV2
BATCH_SIZE = 32
EPOCHS = 50

## 3. Xây dựng Data Pipeline với Augmentation

In [ ]:
# Data Augmentation cho training
def create_augmentation():
    return A.Compose([
        A.RandomBrightnessContrast(p=0.5),
        A.RandomGamma(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.5),
        A.GaussNoise(p=0.3),
        A.Blur(blur_limit=3, p=0.3),
        A.CLAHE(p=0.3),  # Tăng độ tương phản thích ứng
        A.RandomShadow(p=0.3),  # Mô phỏng bóng trong xe
    ])

augmentation = create_augmentation()

In [ ]:
# Tạo data generator
datagen_train = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.5, 1.5],
    horizontal_flip=True,
    validation_split=0.2
)

datagen_test = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Load dữ liệu
train_generator = datagen_train.flow_from_directory(
    'dataset/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = datagen_test.flow_from_directory(
    'dataset/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

## 4. Xây dựng các mô hình

### 4.1. MobileNetV2 (Transfer Learning)

In [ ]:
def create_mobilenet_model():
    # Load pre-trained MobileNetV2
    base_model = MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze base model layers
    base_model.trainable = False
    
    # Tạo model
    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    return model, base_model

mobilenet_model, base_model = create_mobilenet_model()
mobilenet_model.summary()

### 4.2. Custom Lightweight CNN

In [ ]:
def create_custom_cnn():
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(*IMG_SIZE, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        # Block 3 - Depthwise Separable Conv (lightweight)
        layers.SeparableConv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        # Block 4
        layers.SeparableConv2D(256, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        
        # Classifier
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    
    return model

custom_model = create_custom_cnn()
custom_model.summary()

## 5. Training với các kỹ thuật tối ưu

In [ ]:
# Compile model
def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
    )
    return model

# Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint(
        'best_model.h5',
        save_best_only=True,
        monitor='val_accuracy',
        mode='max'
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=0.00001
    ),
    keras.callbacks.TensorBoard(
        log_dir='logs',
        histogram_freq=1
    )
]

In [ ]:
# Train MobileNetV2
mobilenet_model = compile_model(mobilenet_model)

history_mobilenet = mobilenet_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks
)

In [ ]:
# Fine-tuning: Unfreeze một số layer cuối của base model
base_model.trainable = True
fine_tune_at = 100  # Unfreeze từ layer 100 trở đi

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Re-compile với learning rate thấp hơn
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Continue training
history_fine = mobilenet_model.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator,
    callbacks=callbacks
)

## 6. Chuyển đổi sang TensorFlow Lite cho Raspberry Pi

In [ ]:
# Quantization cho TFLite
def convert_to_tflite(model, quantize=True):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    if quantize:
        # INT8 Quantization
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.int8]
        
        # Representative dataset cho calibration
        def representative_dataset():
            for _ in range(100):
                data = np.random.rand(1, *IMG_SIZE, 3).astype(np.float32)
                yield [data]
        
        converter.representative_dataset = representative_dataset
    
    tflite_model = converter.convert()
    return tflite_model

# Convert models
tflite_model = convert_to_tflite(mobilenet_model, quantize=True)
tflite_model_fp16 = convert_to_tflite(mobilenet_model, quantize=False)

# Lưu models
with open('model_int8.tflite', 'wb') as f:
    f.write(tflite_model)

with open('model_fp16.tflite', 'wb') as f:
    f.write(tflite_model_fp16)

print(f"Model INT8 size: {len(tflite_model) / 1024 / 1024:.2f} MB")
print(f"Model FP16 size: {len(tflite_model_fp16) / 1024 / 1024:.2f} MB")

## 7. Chuyển đổi sang ONNX (Tùy chọn)

In [ ]:
!pip install -q tf2onnx onnx onnxruntime

import tf2onnx
import onnx

# Convert to ONNX
spec = (tf.TensorSpec((None, *IMG_SIZE, 3), tf.float32, name="input"),)
output_path = "model.onnx"

model_proto, _ = tf2onnx.convert.from_keras(
    mobilenet_model,
    input_signature=spec,
    opset=13,
    output_path=output_path
)

print(f"ONNX model saved to {output_path}")
print(f"ONNX model size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

## 8. Đánh giá và Visualize kết quả

In [ ]:
# Plot training history
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].set_title('Model Accuracy')
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].set_title('Model Loss')
    
    plt.tight_layout()
    plt.show()

plot_history(history_mobilenet)

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Predict on validation set
predictions = mobilenet_model.predict(validation_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = validation_generator.classes

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASSES))

## 9. Test với ảnh mới

In [ ]:
def predict_image(model, img_path):
    # Load và preprocess ảnh
    img = Image.open(img_path).convert('RGB')
    img = img.resize(IMG_SIZE)
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predict
    predictions = model.predict(img_array)
    predicted_class = CLASSES[np.argmax(predictions)]
    confidence = np.max(predictions) * 100
    
    # Visualize
    plt.figure(figsize=(8, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f'Predicted: {predicted_class}\nConfidence: {confidence:.2f}%')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.bar(CLASSES, predictions[0])
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Probability')
    plt.title('Class Probabilities')
    
    plt.tight_layout()
    plt.show()
    
    return predicted_class, confidence

# Test với ảnh
# test_image_path = 'path/to/test/image.jpg'
# predict_image(mobilenet_model, test_image_path)

## 10. Lưu models và tải về

In [ ]:
# Tạo thư mục output
output_dir = 'trained_models'
os.makedirs(output_dir, exist_ok=True)

# Lưu các models
mobilenet_model.save(f'{output_dir}/mobilenet_model.h5')
shutil.copy('model_int8.tflite', f'{output_dir}/model_int8.tflite')
shutil.copy('model_fp16.tflite', f'{output_dir}/model_fp16.tflite')
shutil.copy('model.onnx', f'{output_dir}/model.onnx')

# Lưu class labels
with open(f'{output_dir}/classes.txt', 'w') as f:
    for cls in CLASSES:
        f.write(f"{cls}\n")

# Zip tất cả
shutil.make_archive('driver_safety_models', 'zip', output_dir)

# Download
files.download('driver_safety_models.zip')

# Copy to Drive
!cp driver_safety_models.zip /content/drive/MyDrive/
print("Models saved to Google Drive!")

## 11. Benchmark Performance

In [ ]:
import time

def benchmark_model(model_path, model_type='tflite', num_runs=100):
    if model_type == 'tflite':
        interpreter = tf.lite.Interpreter(model_path=model_path)
        interpreter.allocate_tensors()
        input_details = interpreter.get_input_details()
        output_details = interpreter.get_output_details()
        
        # Warm up
        for _ in range(10):
            test_input = np.random.rand(1, *IMG_SIZE, 3).astype(np.float32)
            interpreter.set_tensor(input_details[0]['index'], test_input)
            interpreter.invoke()
        
        # Benchmark
        times = []
        for _ in range(num_runs):
            test_input = np.random.rand(1, *IMG_SIZE, 3).astype(np.float32)
            start = time.time()
            interpreter.set_tensor(input_details[0]['index'], test_input)
            interpreter.invoke()
            output = interpreter.get_tensor(output_details[0]['index'])
            times.append(time.time() - start)
        
        avg_time = np.mean(times) * 1000  # Convert to ms
        fps = 1000 / avg_time
        
        print(f"Model: {model_path}")
        print(f"Average inference time: {avg_time:.2f} ms")
        print(f"FPS: {fps:.2f}")
        print(f"Model size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB\n")

# Benchmark các models
benchmark_model('model_int8.tflite')
benchmark_model('model_fp16.tflite')

## Tips để cải thiện performance:\n\n1. **Data Collection**: Thu thập thêm dữ liệu đa dạng trong nhiều điều kiện ánh sáng\n2. **Class Balancing**: Đảm bảo số lượng mẫu cân bằng giữa các class\n3. **Hard Negative Mining**: Thêm các mẫu khó để model học tốt hơn\n4. **Model Ensemble**: Kết hợp nhiều models để tăng độ chính xác\n5. **Edge TPU**: Sử dụng Google Coral hoặc Intel NCS2 để tăng tốc inference